# 🤖 Traductor de Español a Inglés con Transformer (Encoder-Decoder)

Este cuadernillo contiene una implementación clara, directa y funcional de un modelo de **Traducción Automática (Seq2Seq)** de **Español a Inglés** utilizando la arquitectura **Transformer (Encoder-Decoder)** en **PyTorch**.

--- 
## 📌 Estructura del Cuadernillo:
1. **Importación de Librerías y Configuración de Dispositivo (CPU/GPU)**
2. **Creación del Dataset Bilingüe y Procesamiento de Vocabulario**
3. **Codificación Posicional (Positional Encoding)**
4. **Definición de la Arquitectura Transformer (Encoder + Decoder)**
5. **Manejo de Máscaras y Preprocesamiento de Datos**
6. **Entrenamiento del Modelo**
7. **Evaluación e Inferencia (Generación de Traducciones)**

## 1. Importación de Librerías y Configuración de Dispositivo

In [1]:
import math
import torch
import torch.nn as nn
import torch.optim as optim

# Selección del dispositivo de cómputo (GPU si está disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo seleccionado: {device}")

Dispositivo seleccionado: cuda


## 2. Dataset Bilingüe y Construcción del Vocabulario

Creamos un conjunto de datos bilingüe (Español -> Inglés) y tokenizadores simples a nivel de palabras que construyen un índice para los vocabularios de ambos idiomas.

Tokens especiales:
- `<pad>`: Relleno para igualar longitudes de secuencias en un batch (ID: 0).
- `<sos>`: Start of Sentence (Inicio de oración, ID: 1).
- `<eos>`: End of Sentence (Fin de oración, ID: 2).
- `<unk>`: Unknown token (Palabras fuera del vocabulario, ID: 3).

In [2]:
# Dataset sintético de pares de traducción (Español -> Inglés)
data_pairs = [
    ("hola como estas", "hello how are you"),
    ("buenos dias", "good morning"),
    ("buenas noches", "good night"),
    ("me gusta aprender inteligencia artificial", "i like learning artificial intelligence"),
    ("el perro corre en el parque", "the dog runs in the park"),
    ("el gato duerme en la cama", "the cat sleeps on the bed"),
    ("el sol brilla en el cielo", "the sun shines in the sky"),
    ("mi hermano come una manzana", "my brother eats an apple"),
    ("nosotros estudiamos en la universidad", "we study at the university"),
    ("ella lee un libro interesante", "she reads an interesting book"),
    ("el agua esta muy fria", "the water is very cold"),
    ("me gusta comer pizza y pasta", "i like to eat pizza and pasta"),
    ("hoy es un dia hermoso", "today is a beautiful day"),
    ("yo tengo un coche rojo", "i have a red car"),
    ("donde esta el baño", "where is the bathroom"),
    ("gracias por tu ayuda", "thank you for your help"),
    ("el libro esta sobre la mesa", "the book is on the table"),
    ("la musica suena muy bien", "the music sounds very good"),
    ("ellos juegan al futbol", "they play soccer"),
    ("que hora es", "what time is it")
]

class Vocabulary:
    def __init__(self, name):
        self.name = name
        self.PAD_token = 0
        self.SOS_token = 1
        self.EOS_token = 2
        self.UNK_token = 3
        
        self.word2index = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
        self.index2word = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"}
        self.n_words = 4

    def add_sentence(self, sentence):
        for word in sentence.strip().lower().split():
            if word not in self.word2index:
                self.word2index[word] = self.n_words
                self.index2word[self.n_words] = word
                self.n_words += 1

    def numericalize(self, sentence):
        tokens = [self.word2index.get(w, self.UNK_token) for w in sentence.strip().lower().split()]
        return [self.SOS_token] + tokens + [self.EOS_token]

# Instanciar y poblar vocabularios
vocab_es = Vocabulary("Español")
vocab_en = Vocabulary("Inglés")

for es, en in data_pairs:
    vocab_es.add_sentence(es)
    vocab_en.add_sentence(en)

print(f"Vocabulario Español: {vocab_es.n_words} palabras únicas")
print(f"Vocabulario Inglés: {vocab_en.n_words} palabras únicas")

Vocabulario Español: 74 palabras únicas
Vocabulario Inglés: 72 palabras únicas


## 3. Codificación Posicional (Positional Encoding)

Como la arquitectura Transformer procesa todas las palabras en paralelo (sin recurrencia como las RNNs), necesita una forma de conocer el orden y posición de cada palabra dentro de la secuencia. 

Utilizamos la fórmula estándar basada en funciones seno y coseno:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        return x + self.pe[:, :x.size(1)]

## 4. Arquitectura del Modelo: Seq2Seq Transformer (Encoder-Decoder)

El modelo consta de:
1. **Encoder**: Procesa la secuencia de entrada en Español (`src`) aplicando Self-Attention Multi-Cabezal para generar representaciones contextuales integrales.
2. **Decoder**: Genera autoregresivamente la secuencia objetivo en Inglés (`tgt`) utilizando tanto Self-Attention Causal (para no ver tokens futuros) como Cross-Attention sobre la salida del Encoder.
3. **Proyección Lineal Final**: Mapea la representación latente de dimensión `d_model` a distribuciones de probabilidad sobre el vocabulario de destino.

In [4]:
class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=128, nhead=4, 
                 num_encoder_layers=2, num_decoder_layers=2, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        # Capas de Embedding + Codificación Posicional
        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=0)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model)
        
        # Núcleo Transformer Encoder-Decoder de PyTorch
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        
        # Capa lineal proyectora a vocabulario objetivo
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt):
        tgt_seq_len = tgt.size(1)
        
        # Máscara causal para impedir que el decoder mire palabras futuras durante la enseñanza (Teacher Forcing)
        tgt_mask = self.transformer.generate_square_subsequent_mask(tgt_seq_len).to(tgt.device)
        
        # Máscaras de relleno (padding) de clave
        src_key_padding_mask = (src == 0)
        tgt_key_padding_mask = (tgt == 0)
        
        # Aplicar embeddings escalados por sqrt(d_model) y codificación posicional
        src_emb = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoder(self.tgt_embedding(tgt) * math.sqrt(self.d_model))
        
        # Forward a través del Transformer Encoder-Decoder
        out = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )
        
        return self.fc_out(out)

## 5. Preprocesamiento de Datos y Formato de Lotes (Batches)

In [5]:
# Función para rellenar secuencias con token <pad> hasta la longitud máxima en el batch
def pad_sequences(sequences, pad_val=0):
    max_len = max(len(seq) for seq in sequences)
    padded = [seq + [pad_val] * (max_len - len(seq)) for seq in sequences]
    return torch.tensor(padded, dtype=torch.long)

# Numeralizar frases
src_seqs = [vocab_es.numericalize(es) for es, en in data_pairs]
tgt_seqs = [vocab_en.numericalize(en) for es, en in data_pairs]

src_tensor = pad_sequences(src_seqs).to(device)
tgt_tensor = pad_sequences(tgt_seqs).to(device)

print(f"Dimensión Tensor Origen (Español): {src_tensor.shape}")
print(f"Dimensión Tensor Destino (Inglés):  {tgt_tensor.shape}")

Dimensión Tensor Origen (Español): torch.Size([20, 8])
Dimensión Tensor Destino (Inglés):  torch.Size([20, 9])


## 6. Entrenamiento del Modelo Transformer

Entrenamos el modelo utilizando **Teacher Forcing**:
- `tgt_input`: La secuencia objetivo de entrada para el decoder (desde `<sos>` hasta el penúltimo token).
- `tgt_output`: La secuencia esperada como salida del decoder (desde el primer token real hasta `<eos>`).
- Pérdida: `CrossEntropyLoss` ignorando el token de relleno `<pad>` (`ignore_index=0`).

In [6]:
# Instanciar modelo
model = Seq2SeqTransformer(
    src_vocab_size=vocab_es.n_words, 
    tgt_vocab_size=vocab_en.n_words,
    d_model=128,
    nhead=4,
    num_encoder_layers=2,
    num_decoder_layers=2,
    dim_feedforward=256,
    dropout=0.1
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Bucle de Entrenamiento
model.train()
epochs = 200

print("Iniciando entrenamiento...")
for epoch in range(1, epochs + 1):
    optimizer.zero_grad()
    
    tgt_input = tgt_tensor[:, :-1]
    tgt_output = tgt_tensor[:, 1:]
    
    output = model(src_tensor, tgt_input)
    
    # Reformatear salida para calcular la pérdida sobre el vocabulario objetivo
    loss = criterion(output.reshape(-1, vocab_en.n_words), tgt_output.reshape(-1))
    
    loss.backward()
    optimizer.step()
    
    if epoch % 25 == 0 or epoch == 1:
        print(f"Época {epoch:3d}/{epochs} | Pérdida (Loss): {loss.item():.4f}")

Iniciando entrenamiento...


c:\Users\ASUS TUF GAMING A15\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Época   1/200 | Pérdida (Loss): 4.5505
Época  25/200 | Pérdida (Loss): 0.6892
Época  50/200 | Pérdida (Loss): 0.1158
Época  75/200 | Pérdida (Loss): 0.0439
Época 100/200 | Pérdida (Loss): 0.0274
Época 125/200 | Pérdida (Loss): 0.0193
Época 150/200 | Pérdida (Loss): 0.0140
Época 175/200 | Pérdida (Loss): 0.0131
Época 200/200 | Pérdida (Loss): 0.0100


## 7. Inferencia y Traducción Interactiva (Greedy Search)

En la etapa de inferencia (traducción de frases nuevas):
1. El Encoder procesa la frase en Español completa.
2. El Decoder comienza únicamente con el token `<sos>`.
3. En cada paso paso a paso (autoregresivamente), se predice el token más probable (`argmax`) y se agrega a la secuencia objetivo hasta generar `<eos>` o alcanzar la longitud máxima.

In [7]:
def translate_sentence(sentence, max_len=20):
    model.eval()
    with torch.no_grad():
        # 1. Numeralizar la frase de entrada en español
        src_tokens = vocab_es.numericalize(sentence)
        src_tensor = torch.tensor([src_tokens], dtype=torch.long).to(device)
        
        # 2. Inicializar la secuencia objetivo del decoder con <sos>
        tgt_tokens = [vocab_en.SOS_token]
        
        # 3. Decodificación autoregresiva paso a paso
        for _ in range(max_len):
            tgt_tensor = torch.tensor([tgt_tokens], dtype=torch.long).to(device)
            output = model(src_tensor, tgt_tensor)
            
            # Obtener la palabra más probable del último paso de tiempo predicho
            next_token = output[0, -1, :].argmax().item()
            tgt_tokens.append(next_token)
            
            if next_token == vocab_en.EOS_token:
                break
                
        # 4. Convertir índices a palabras (excluyendo tokens especiales <sos> y <eos>)
        translated_words = [
            vocab_en.index2word[idx] 
            for idx in tgt_tokens 
            if idx not in (vocab_en.SOS_token, vocab_en.EOS_token)
        ]
        return " ".join(translated_words)

# Pruebas de traducción con frases del dataset e inferencias nuevas
test_sentences = [
    "hola como estas",
    "el sol brilla en el cielo",
    "me gusta aprender inteligencia artificial",
    "ella lee un libro interesante",
    "donde esta el baño",
    "buenos dias",
    "gracias por tu ayuda"
]

print("="*55)
print(f"{'ESPAÑOL':<40} -> {'TRADUCCIÓN AL INGLÉS'}")
print("="*55)
for sentence in test_sentences:
    translation = translate_sentence(sentence)
    print(f"{sentence:<40} -> {translation}")
print("="*55)

ESPAÑOL                                  -> TRADUCCIÓN AL INGLÉS


c:\Users\ASUS TUF GAMING A15\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\transformer.py:508: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


hola como estas                          -> hello how are you
el sol brilla en el cielo                -> the sun shines in the sky
me gusta aprender inteligencia artificial -> i like learning artificial intelligence
ella lee un libro interesante            -> she reads an interesting book
donde esta el baño                       -> where is the bathroom
buenos dias                              -> good morning
gracias por tu ayuda                     -> thank you for your help
